# Ablation Study v2: Physics-Feature Parameterizations

Train on first 16 LHS beams (or 5 random 12-of-16 subsets). For each config,
generate a single next-beam recommendation, then score it.

Ground truth GP: `b_dH_Pltb_Pbend` (4D, Matern) trained on all beams.

Scoring metrics:
1. `dist_to_strong` -- Euclidean distance in normalized (b,H) space to nearest tested beam with Str/w >= 31.75
2. `gt_strw` -- predicted Str/w from the ground-truth GP at the recommended point
3. `gt_var` -- GP predictive variance at that point (confidence in the GT prediction)

Parameterizations tested:
1. `b_H` -- raw design variables
2. `b_dH_R` -- web thickness + deviation from optimal H + stability ratio
3. `Pltb_Pbend` -- normalized LTB and bending strength per unit mass
4. `b_Pltb_Pbend` -- web thickness + both normalized strengths
5. `b_dH_Pltb_Pbend` -- web thickness + dH + both normalized strengths

J uses Timoshenko correction throughout (b < H and h < B assumed).
No ls_bounds variation (adaptive only).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, ConstantKernel
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import norm
from itertools import combinations
import warnings
import time
warnings.filterwarnings("ignore")

In [ ]:
# =====================================================================
# PHYSICS -- Sunlu PLA+ 2.0 (ASTM D790)
# =====================================================================
TOTAL_HEIGHT = 25.0       # mm
B_FIXED      = 16.0       # mm
LENGTH_M     = 0.2023     # m, span
SIGMA_Y      = 81.8e6     # Pa
E_MOD        = 2.74e9     # Pa
G_MOD        = E_MOD / 2.6
C1           = 1.35       # moment gradient factor, 3-pt bending
DENSITY      = 1210       # kg/m^3
Y_MAX        = TOTAL_HEIGHT / 2e3  # m, NA to extreme fiber

# --- Section properties (all inputs mm, outputs m^4 or kg) ---

def calc_Ix(H, b, B=B_FIXED):
    """Strong-axis I (m^4). h derived internally."""
    H_m, b_m, B_m = H/1e3, b/1e3, B/1e3
    h_m = (TOTAL_HEIGHT/1e3 - H_m) / 2.0
    if h_m <= 0:
        return 0.0
    return (b_m * H_m**3)/12 + 2*(B_m * h_m**3/12 + B_m * h_m * ((H_m + h_m)/2)**2)

def calc_Iy(H, b, B=B_FIXED):
    """Weak-axis I (m^4)."""
    H_m, b_m, B_m = H/1e3, b/1e3, B/1e3
    h_m = (TOTAL_HEIGHT/1e3 - H_m) / 2.0
    if h_m <= 0:
        return 0.0
    return (H_m * b_m**3)/12 + 2*(h_m * B_m**3)/12

def calc_J(H, b, B=B_FIXED):
    """Torsional constant, Timoshenko correction (m^4).
    Assumes b < H and h < B."""
    H_m, b_m, B_m = H/1e3, b/1e3, B/1e3
    h_m = (TOTAL_HEIGHT/1e3 - H_m) / 2.0
    if h_m <= 0:
        return 0.0
    rw = b_m / H_m
    beta_w = (1.0/3.0) * (1.0 - 0.63*rw + 0.052*rw**5)
    J_web = beta_w * H_m * b_m**3
    rf = h_m / B_m
    beta_f = (1.0/3.0) * (1.0 - 0.63*rf + 0.052*rf**5)
    J_fl = beta_f * B_m * h_m**3
    return J_web + 2 * J_fl

def calc_mass(H, b, B=B_FIXED):
    """Beam mass in grams."""
    H_m, b_m, B_m = H/1e3, b/1e3, B/1e3
    h_m = (TOTAL_HEIGHT/1e3 - H_m) / 2.0
    if h_m <= 0:
        return 0.0
    return DENSITY * LENGTH_M * (H_m * b_m + 2 * h_m * B_m) * 1000

def calc_P_bend(H, b):
    """First-yield failure load from bending (N)."""
    Ix = calc_Ix(H, b)
    My = SIGMA_Y * Ix / Y_MAX
    return 4 * My / LENGTH_M

def calc_P_ltb(H, b):
    """Elastic critical LTB load (N)."""
    Iy = calc_Iy(H, b)
    J  = calc_J(H, b)
    if Iy <= 0 or J <= 0:
        return 0.0
    Mcr = (C1 * np.pi / LENGTH_M) * np.sqrt(E_MOD * Iy * G_MOD * J)
    return 4 * Mcr / LENGTH_M

def calc_R(H, b):
    """Stability ratio Mcr / My."""
    Ix = calc_Ix(H, b)
    My = SIGMA_Y * Ix / Y_MAX
    if My <= 0:
        return 0.0
    Iy = calc_Iy(H, b)
    J  = calc_J(H, b)
    if Iy <= 0 or J <= 0:
        return 0.0
    Mcr = (C1 * np.pi / LENGTH_M) * np.sqrt(E_MOD * Iy * G_MOD * J)
    return Mcr / My

def calc_str_w(H, b):
    """Bending-based Str/w (N/g)."""
    P = calc_P_bend(H, b)
    m = calc_mass(H, b)
    return P / m if m > 0 else 0.0

def find_H_opt(b):
    """H that maximizes bending Str/w for a given b."""
    def obj(H):
        if H < 12.0 or H > 23.4:
            return 1e10
        h = (TOTAL_HEIGHT - H) / 2.0
        if h < 0 or h > 6.5:
            return 1e10
        return -calc_str_w(H, b)
    return minimize_scalar(obj, bounds=(12.0, 23.4), method='bounded').x

In [ ]:
# =====================================================================
# DATA
# =====================================================================
df_all = pd.read_csv('../data/I_beam_data_2var.csv')
b_all = df_all['b_web_mm'].values.astype(float)
H_all = df_all['H_web_mm'].values.astype(float)
y_all = df_all['Str/w N/g'].values.astype(float)

print(f'Total beams: {len(df_all)}')
print(f'First 16 (LHS set): beams {list(df_all["Beam Number"].iloc[:16].values)}')

In [ ]:
# =====================================================================
# PARAMETERIZATION TRANSFORMS
# =====================================================================
# Each transform: (b_arr, H_arr) -> X array, plus bounds dict.
# The bounds are used for normalization to [0,1].

def _Pltb_per_mass(b, H):
    m = calc_mass(H, b)
    return calc_P_ltb(H, b) / m if m > 0 else 0.0

def _Pbend_per_mass(b, H):
    m = calc_mass(H, b)
    return calc_P_bend(H, b) / m if m > 0 else 0.0

def _dH(b, H):
    return H - find_H_opt(b)

def _R(b, H):
    return calc_R(H, b)

def to_bH(b_arr, H_arr):
    return np.column_stack([b_arr, H_arr])

def to_bdHR(b_arr, H_arr):
    n = len(b_arr)
    out = np.zeros((n, 3))
    for i in range(n):
        out[i, 0] = b_arr[i]
        out[i, 1] = _dH(b_arr[i], H_arr[i])
        out[i, 2] = _R(b_arr[i], H_arr[i])
    return out

def to_Pltb_Pbend(b_arr, H_arr):
    n = len(b_arr)
    out = np.zeros((n, 2))
    for i in range(n):
        out[i, 0] = _Pltb_per_mass(b_arr[i], H_arr[i])
        out[i, 1] = _Pbend_per_mass(b_arr[i], H_arr[i])
    return out

def to_b_Pltb_Pbend(b_arr, H_arr):
    n = len(b_arr)
    out = np.zeros((n, 3))
    for i in range(n):
        out[i, 0] = b_arr[i]
        out[i, 1] = _Pltb_per_mass(b_arr[i], H_arr[i])
        out[i, 2] = _Pbend_per_mass(b_arr[i], H_arr[i])
    return out

def to_b_dH_Pltb_Pbend(b_arr, H_arr):
    n = len(b_arr)
    out = np.zeros((n, 4))
    for i in range(n):
        out[i, 0] = b_arr[i]
        out[i, 1] = _dH(b_arr[i], H_arr[i])
        out[i, 2] = _Pltb_per_mass(b_arr[i], H_arr[i])
        out[i, 3] = _Pbend_per_mass(b_arr[i], H_arr[i])
    return out

# Compute bounds from the full 16-beam LHS set (with margin)
b16, H16 = b_all[:16], H_all[:16]
_X_bdHR_16 = to_bdHR(b16, H16)
_X_PP_16   = to_Pltb_Pbend(b16, H16)
_X_bPP_16  = to_b_Pltb_Pbend(b16, H16)
_X_bdHPP_16 = to_b_dH_Pltb_Pbend(b16, H16)

def _bounds_from_data(X, names, margin=0.15):
    """Compute bounds from data columns with fractional margin."""
    bds = {}
    for i, nm in enumerate(names):
        lo, hi = X[:, i].min(), X[:, i].max()
        span = hi - lo
        if span < 1e-9:
            span = abs(lo) * 0.1 + 1e-6
        bds[nm] = (lo - margin * span, hi + margin * span)
    return bds

BOUNDS_bH = {'b': (1.0, 8.0), 'H': (12.0, 23.0)}

BOUNDS_bdHR = _bounds_from_data(_X_bdHR_16, ['b', 'dH', 'R'])
BOUNDS_bdHR['b'] = (1.0, 8.0)  # keep physical range for b

BOUNDS_PP = _bounds_from_data(_X_PP_16, ['Pltb_m', 'Pbend_m'])
BOUNDS_bPP = _bounds_from_data(_X_bPP_16, ['b', 'Pltb_m', 'Pbend_m'])
BOUNDS_bPP['b'] = (1.0, 8.0)

BOUNDS_bdHPP = _bounds_from_data(_X_bdHPP_16, ['b', 'dH', 'Pltb_m', 'Pbend_m'])
BOUNDS_bdHPP['b'] = (1.0, 8.0)

PARAM_CONFIGS = {
    'b_H':            {'transform': to_bH,              'bounds': BOUNDS_bH,    'ndim': 2},
    'b_dH_R':         {'transform': to_bdHR,            'bounds': BOUNDS_bdHR,  'ndim': 3},
    'Pltb_Pbend':     {'transform': to_Pltb_Pbend,      'bounds': BOUNDS_PP,    'ndim': 2},
    'b_Pltb_Pbend':   {'transform': to_b_Pltb_Pbend,    'bounds': BOUNDS_bPP,   'ndim': 3},
    'b_dH_Pltb_Pbend':{'transform': to_b_dH_Pltb_Pbend, 'bounds': BOUNDS_bdHPP, 'ndim': 4},
}

print('Parameterizations and dimensions:')
for k, v in PARAM_CONFIGS.items():
    print(f'  {k:20s}  {v["ndim"]}D  bounds: {list(v["bounds"].keys())}')

In [ ]:
# =====================================================================
# GP TRAINING
# =====================================================================

def normalize(X, bounds):
    X_n = np.copy(X).astype(float)
    keys = list(bounds.keys())
    for i, k in enumerate(keys):
        lo, hi = bounds[k]
        X_n[:, i] = (X[:, i] - lo) / (hi - lo)
    return X_n

def denormalize(X_n, bounds):
    X = np.copy(X_n)
    keys = list(bounds.keys())
    for i, k in enumerate(keys):
        lo, hi = bounds[k]
        X[:, i] = X_n[:, i] * (hi - lo) + lo
    return X

def train_gp(X, y, bounds, kernel_type='matern', alpha=1e-4):
    """Train GP in log-space. Returns (gp, X_norm, y_centered, y_mean)."""
    X_n = normalize(X, bounds)
    y_log = np.log(y)
    y_mean = np.mean(y_log)
    y_c = y_log - y_mean
    nd = X.shape[1]
    ls_bounds = (0.15, 3.0)
    if kernel_type == 'matern':
        base = Matern(length_scale=[0.5]*nd, length_scale_bounds=ls_bounds, nu=2.5)
    else:
        base = RBF(length_scale=[0.5]*nd, length_scale_bounds=ls_bounds)
    kernel = ConstantKernel(1.0) * base
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15,
                                  alpha=alpha, normalize_y=False)
    gp.fit(X_n, y_c)
    return gp, X_n, y_c, y_mean

In [ ]:
# =====================================================================
# ACQUISITION -- candidate-based in physical (b, H) space
# =====================================================================

def recommend_in_bH(gp, y_c, y_mean, bounds, transform_fn,
                    acq_type='ei', kappa=2.0, n_cand=5000):
    """
    Generate random (b, H) candidates, transform to param space,
    score with GP acquisition, return best (b, H, predicted Str/w).
    """
    b_cand = np.random.uniform(BOUNDS_bH['b'][0], BOUNDS_bH['b'][1], n_cand)
    H_cand = np.random.uniform(BOUNDS_bH['H'][0], BOUNDS_bH['H'][1], n_cand)
    X_cand = transform_fn(b_cand, H_cand)
    X_cand_n = normalize(X_cand, bounds)
    mu, sig = gp.predict(X_cand_n, return_std=True)

    if acq_type == 'ei':
        best_y = np.max(y_c)
        Z = np.where(sig > 1e-9, (mu - best_y) / sig, 0.0)
        acq = np.where(sig > 1e-9,
                       (mu - best_y) * norm.cdf(Z) + sig * norm.pdf(Z),
                       0.0)
    else:
        acq = mu + kappa * sig

    idx = np.argmax(acq)
    pred_strw = np.exp(mu[idx] + y_mean)
    return b_cand[idx], H_cand[idx], pred_strw

In [ ]:
# =====================================================================
# GROUND TRUTH GP: b_dH_Pltb_Pbend on all beams
# =====================================================================

STRONG_THRESHOLD = 31.75  # Str/w N/g

def build_ground_truth():
    X_gt = to_b_dH_Pltb_Pbend(b_all, H_all)
    gt_bounds = _bounds_from_data(X_gt, ['b', 'dH', 'Pltb_m', 'Pbend_m'])
    gt_bounds['b'] = (1.0, 8.0)
    gp_gt, _, _, y_gt_mean = train_gp(
        X_gt, y_all, gt_bounds, kernel_type='matern', alpha=3e-5)
    return gp_gt, gt_bounds, y_gt_mean

def eval_gt(gp_gt, gt_bounds, y_gt_mean, b_rec, H_rec):
    """Predict Str/w and variance at (b_rec, H_rec) using the GT GP."""
    X_pt = to_b_dH_Pltb_Pbend(np.array([b_rec]), np.array([H_rec]))
    X_pt_n = normalize(X_pt, gt_bounds)
    mu, sig = gp_gt.predict(X_pt_n, return_std=True)
    gt_strw = np.exp(mu[0] + y_gt_mean)
    gt_var = sig[0]**2
    return gt_strw, gt_var

def dist_to_strong(b_rec, H_rec):
    """Min Euclidean distance in normalized (b,H) to a tested beam with Str/w >= STRONG_THRESHOLD."""
    mask = y_all >= STRONG_THRESHOLD
    if not np.any(mask):
        return 1.0
    b_s, H_s = b_all[mask], H_all[mask]
    b_n = (b_rec - 1.0) / 7.0
    H_n = (H_rec - 12.0) / 11.0
    b_sn = (b_s - 1.0) / 7.0
    H_sn = (H_s - 12.0) / 11.0
    return np.min(np.sqrt((b_n - b_sn)**2 + (H_n - H_sn)**2))

In [ ]:
# =====================================================================
# TRAINING SUBSETS: full 16 + 5 random 12-of-16 subsets
# =====================================================================

np.random.seed(42)

all_16_idx = np.arange(16)

subsets = [('all_16', all_16_idx)]
for s in range(5):
    drop4 = np.random.choice(16, size=4, replace=False)
    keep = np.array(sorted(set(range(16)) - set(drop4)))
    subsets.append((f'sub12_{s}', keep))

print('Training subsets:')
for name, idx in subsets:
    print(f'  {name}: {len(idx)} beams, indices {list(idx)}')

In [ ]:
# =====================================================================
# MAIN ABLATION SWEEP
# =====================================================================

def run_ablation():
    print('Building ground-truth GP (b_dH_Pltb_Pbend on all beams)...')
    gp_gt, gt_bounds, y_gt_mean = build_ground_truth()
    print(f'Strong-zone threshold: Str/w >= {STRONG_THRESHOLD}')
    n_strong = np.sum(y_all >= STRONG_THRESHOLD)
    print(f'Beams above threshold: {n_strong} of {len(y_all)}')

    # Noise baseline from repeated beam (Beam 17 & 24, both b=4.5 H=14.5)
    y_repeat = y_all[[19, 26]]  # indices for beams 17 and 24
    baseline_var = np.var(np.log(y_repeat), ddof=1)
    print(f'Noise baseline (log-space var from repeats): {baseline_var:.2e}')

    parameterizations = list(PARAM_CONFIGS.keys())
    kernels = ['matern', 'rbf']
    acquisitions = [('ei', None), ('ucb', 1.0), ('ucb', 2.0), ('ucb', 3.0)]
    noise_levels = [baseline_var, 1e-4, 3e-4, 1e-3, 3e-3]

    total = (len(subsets) * len(parameterizations) * len(kernels) *
             len(acquisitions) * len(noise_levels))
    print(f'Total configs: {total}')
    print('Running...\n')

    results = []
    done = 0
    t0 = time.time()

    for sub_name, sub_idx in subsets:
        b_tr = b_all[sub_idx]
        H_tr = H_all[sub_idx]
        y_tr = y_all[sub_idx]

        for pname in parameterizations:
            cfg = PARAM_CONFIGS[pname]
            X_tr = cfg['transform'](b_tr, H_tr)
            bds = cfg['bounds']

            for kern in kernels:
                for acq_type, kappa in acquisitions:
                    for noise in noise_levels:
                        done += 1
                        if done % 50 == 0:
                            elapsed = time.time() - t0
                            print(f'  {done}/{total}  ({elapsed:.0f}s)')

                        try:
                            gp, X_n, y_c, y_mean = train_gp(
                                X_tr, y_tr, bds,
                                kernel_type=kern, alpha=noise)

                            b_rec, H_rec, pred = recommend_in_bH(
                                gp, y_c, y_mean, bds, cfg['transform'],
                                acq_type=acq_type,
                                kappa=kappa if kappa else 2.0)

                            gt_strw, gt_var = eval_gt(gp_gt, gt_bounds, y_gt_mean, b_rec, H_rec)
                            d_strong = dist_to_strong(b_rec, H_rec)
                            ls = gp.kernel_.k2.length_scale

                            results.append({
                                'subset': sub_name,
                                'param': pname,
                                'kernel': kern,
                                'acquisition': acq_type,
                                'kappa': kappa,
                                'noise': noise,
                                'b_rec': round(b_rec, 3),
                                'H_rec': round(H_rec, 3),
                                'pred_strw': round(pred, 2),
                                'gt_strw': round(gt_strw, 2),
                                'gt_var': round(gt_var, 6),
                                'dist_to_strong': round(d_strong, 4),
                                'in_strong_zone': d_strong < 0.15,
                                'length_scales': [round(x, 3) for x in ls],
                            })

                        except Exception as e:
                            results.append({
                                'subset': sub_name,
                                'param': pname,
                                'kernel': kern,
                                'acquisition': acq_type,
                                'kappa': kappa,
                                'noise': noise,
                                'error': str(e),
                            })

    elapsed = time.time() - t0
    print(f'\nDone in {elapsed:.1f}s')
    return pd.DataFrame(results)

In [ ]:
np.random.seed(42)
df_results = run_ablation()
df_results.to_csv('ablation_results_2var_v2.csv', index=False)
print(f'Saved {len(df_results)} results to ablation_results_2var_v2.csv')

In [ ]:
# =====================================================================
# ANALYSIS
# =====================================================================

ok = df_results[~df_results['gt_strw'].isna()].copy()
print(f'Valid results: {len(ok)} of {len(df_results)}')
print(f'GT model: b_dH_Pltb_Pbend (Matern, all beams)')
print(f'Strong threshold: Str/w >= {STRONG_THRESHOLD}')
print()

print('=== Mean gt_strw by parameterization ===')
print(ok.groupby('param')['gt_strw'].agg(['mean','std','count']).sort_values('mean', ascending=False))
print()

print('=== Mean gt_var (GT confidence) by parameterization ===')
print(ok.groupby('param')['gt_var'].agg(['mean','std']).sort_values('mean'))
print()

print('=== Strong-zone hit rate by parameterization ===')
print(ok.groupby('param')['in_strong_zone'].mean().sort_values(ascending=False))
print()

print('=== Mean gt_strw by param + acquisition ===')
print(ok.groupby(['param','acquisition'])['gt_strw'].mean().sort_values(ascending=False))
print()

print('=== Mean gt_strw by param + subset ===')
print(ok.groupby(['param','subset'])['gt_strw'].mean().unstack('subset').round(2))
print()

print('=== Mean gt_strw by noise level ===')
print(ok.groupby('noise')['gt_strw'].mean())
print()

print('=== Top 15 configs by gt_strw ===')
cols = ['subset','param','kernel','acquisition','kappa','noise',
        'b_rec','H_rec','gt_strw','gt_var','dist_to_strong','in_strong_zone']
print(ok.sort_values('gt_strw', ascending=False)[cols].head(15).to_string(index=False))
print()

print('=== Top 15 configs by dist_to_strong (closest to confirmed strong beam) ===')
print(ok.sort_values('dist_to_strong')[cols].head(15).to_string(index=False))
print()

print('=== Configs with high gt_strw AND low gt_var (confident + strong) ===')
confident = ok[(ok['gt_strw'] >= 31.0) & (ok['gt_var'] < ok['gt_var'].quantile(0.25))].copy()
print(f'{len(confident)} configs meet criteria')
if len(confident) > 0:
    print(confident.sort_values('gt_strw', ascending=False)[cols].head(15).to_string(index=False))